# DengAI: Predicción de la propagación de enfermedades

**Actividad 3.6** - Sistemas de Aprendizaje Automático  
Competición DrivenData: [DengAI - Predicting Disease Spread](https://www.drivendata.org/competitions/44/dengai-predicting-disease-spread/)

---

## Índice
1. Importación del dataset (GitHub/Drive)
2. Preparación y calidad de los datos
3. Selección de características
4. División Train / Validation / Test
5. Modelo 1: Naive Bayes
6. Modelo 2: KNN
7. Modelo 3: Random Forest
8. Comparativa de modelos y gráficos
9. Predicción y validación de resultados
10. Propuestas creativas e innovadoras
11. Generación del fichero de submit

## 1. Importación del dataset

Se utilizan **GitHub** o **Google Drive** como origen de los ficheros. Para ejecución local se usa la carpeta `data/`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración de visualización
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style('whitegrid')

def load_data_from_github(base_url=None):
    """Carga de datos desde URL de GitHub (raw)."""
    if base_url is None:
        return None
    try:
        labels = pd.read_csv(f"{base_url}/dengue_labels_train.csv")
        feat_train = pd.read_csv(f"{base_url}/dengue_features_train.csv")
        feat_test = pd.read_csv(f"{base_url}/dengue_features_test.csv")
        return labels, feat_train, feat_test
    except Exception as e:
        print(f"Error cargando desde GitHub: {e}")
        return None

def load_data_local(data_dir='data'):
    """Carga desde carpeta local (p. ej. data/)."""
    data_path = Path(data_dir)
    labels = pd.read_csv(data_path / 'dengue_labels_train.csv')
    feat_train = pd.read_csv(data_path / 'dengue_features_train.csv')
    feat_test = pd.read_csv(data_path / 'dengue_features_test.csv')
    return labels, feat_train, feat_test

# Origen: carpeta local (para GitHub/Colab sustituir por URL o montar Drive)
DATA_DIR = 'data'
labels_train, features_train, features_test = load_data_local(DATA_DIR)

print("Labels train:", labels_train.shape)
print("Features train:", features_train.shape)
print("Features test:", features_test.shape)
labels_train.head()

### Carga alternativa desde Google Drive (Colab)

```python
# from google.colab import drive
# drive.mount('/content/drive')
# labels_train, features_train, features_test = load_data_local('/content/drive/MyDrive/MOSQUITO/data')
```

### Carga alternativa desde GitHub (URL raw)

```python
# url_github = 'https://raw.githubusercontent.com/TU_USUARIO/TU_REPO/main/data'
# labels_train, features_train, features_test = load_data_from_github(url_github)
```

## 2. Preparación y calidad de los datos

Se normalizan y ajustan los datos: tratamiento de missing, tipos y escalado.

In [ ]:
# Unir features con etiquetas por city, year, weekofyear
df = features_train.merge(labels_train, on=['city', 'year', 'weekofyear'], how='inner')

# Eliminar columnas no numéricas para el modelo (week_start_date)
col_fecha = 'week_start_date'
if col_fecha in df.columns:
    df = df.drop(columns=[col_fecha])
if col_fecha in features_test.columns:
    features_test_clean = features_test.drop(columns=[col_fecha]).copy()
else:
    features_test_clean = features_test.copy()

# Codificar ciudad
df['city'] = (df['city'] == 'sj').astype(int)
features_test_clean['city'] = (features_test_clean['city'] == 'sj').astype(int)

# Detección de valores faltantes (cadenas vacías o NaN)
for c in df.select_dtypes(include=[np.number]).columns:
    df[c] = pd.to_numeric(df[c], errors='coerce')
for c in features_test_clean.select_dtypes(include=[np.number]).columns:
    features_test_clean[c] = pd.to_numeric(features_test_clean[c], errors='coerce')

print("Valores faltantes por columna (train):")
print(df.isnull().sum()[df.isnull().sum() > 0])

# Rellenar faltantes con la mediana por columna (robusto a outliers)
target_col = 'total_cases'
feature_cols = [c for c in df.columns if c != target_col]

medians = df[feature_cols].median()
df[feature_cols] = df[feature_cols].fillna(medians)
features_test_clean[feature_cols] = features_test_clean[feature_cols].fillna(medians)

print("\nTras rellenar faltantes:", df.isnull().sum().sum(), "NaN en train")
df.describe()

In [ ]:
# Normalización: StandardScaler (media 0, varianza 1) para modelos que lo requieren
from sklearn.preprocessing import StandardScaler, MinMaxScaler

X_full = df[feature_cols].copy()
y_full = df[target_col].copy()

scaler = StandardScaler()
X_full_norm = pd.DataFrame(
    scaler.fit_transform(X_full),
    columns=feature_cols,
    index=X_full.index
)
X_test_norm = pd.DataFrame(
    scaler.transform(features_test_clean[feature_cols]),
    columns=feature_cols,
    index=features_test_clean.index
)

print("Datos normalizados (muestra):")
X_full_norm.describe().loc[['mean', 'std']]

## 3. Selección de características

Se utilizan **métodos y gráficos** para la selección: correlación con el target, importancia (árbol) y matriz de correlación.

In [ ]:
# Correlación con la variable objetivo
corr_target = X_full_norm.copy()
corr_target['total_cases'] = y_full
correlations = corr_target.corr()['total_cases'].drop('total_cases').sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
correlations.plot(kind='barh', ax=ax, color='steelblue', edgecolor='navy')
ax.set_title('Selección de características: correlación con total_cases')
ax.set_xlabel('Correlación (Pearson)')
plt.tight_layout()
plt.show()

In [ ]:
# Importancia con árbol de decisión (herramienta gráfica para selección)
from sklearn.tree import DecisionTreeRegressor

dt = DecisionTreeRegressor(random_state=42, max_depth=10)
dt.fit(X_full_norm, y_full)
imp = pd.Series(dt.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
imp.plot(kind='barh', ax=ax, color='darkgreen', alpha=0.8)
ax.set_title('Selección de características: importancia (Decision Tree)')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

In [ ]:
# Matriz de correlación entre predictores (reducir redundancia)
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(X_full_norm.corr(), cmap='RdBu_r', center=0, ax=ax, 
            square=True, linewidths=0.5, fmt='.1f')
ax.set_title('Matriz de correlación entre características (selección de redundantes)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Criterio: mantener características con |correlación con target| > umbral o alta importancia
umbral_corr = 0.05
selected_by_corr = correlations[abs(correlations) >= umbral_corr].index.tolist()
selected_by_imp = imp[imp >= imp.quantile(0.2)].index.tolist()
selected_features = list(dict.fromkeys(selected_by_corr + selected_by_imp))
if not selected_features:
    selected_features = feature_cols
print("Características seleccionadas:", selected_features)

## 4. División Train / Validation / Test

Además de train y test, se incorpora **conjunto de validación** para ajuste de hiperparámetros y selección de modelo.

In [ ]:
from sklearn.model_selection import train_test_split

X = X_full_norm[selected_features].copy()
y = y_full.copy()

# 70% train, 15% validation, 15% test
X_train, X_rest, y_train, y_rest = train_test_split(X, y, test_size=0.30, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_rest, y_rest, test_size=0.5, random_state=42)

print("Train:", X_train.shape[0], "| Validation:", X_val.shape[0], "| Test:", X_test.shape[0])

## 5. Modelo 1: Naive Bayes

Para regresión se discretiza el target en intervalos, se entrena **GaussianNB** y se convierte la predicción de clase a valor numérico (centro del intervalo). Se usan **GridSearch** y **RandomizedSearch** con **Cross Validation** para la selección.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, cross_val_score
from sklearn.metrics import mean_absolute_error
from sklearn.preprocessing import LabelEncoder

# Discretizar total_cases en bins para Naive Bayes (clasificación)
n_bins = 15
y_train_binned = pd.cut(y_train, bins=n_bins, labels=False)
y_val_binned = pd.cut(y_val, bins=n_bins, labels=False)

# Bordes de los bins (para recuperar valor central después)
_, bin_edges = pd.cut(y_train, bins=n_bins, retbins=True)
bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2

def nb_pred_to_regression(clf, X, bin_centers):
    pred_class = clf.predict(X)
    pred_class = np.clip(pred_class, 0, len(bin_centers)-1)
    return bin_centers[pred_class.astype(int)]

# Hiperparámetros limitados para Naive Bayes (var_smoothing)
param_grid_nb = {'var_smoothing': np.logspace(-12, -8, 20)}

grid_nb = GridSearchCV(GaussianNB(), param_grid_nb, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_nb.fit(X_train, y_train_binned)

print("Mejores parámetros (GridSearch):", grid_nb.best_params_)
print("Mejor CV score (MAE):", -grid_nb.best_score_)

y_val_pred_nb = nb_pred_to_regression(grid_nb.best_estimator_, X_val, bin_centers)
mae_nb_val = mean_absolute_error(y_val, y_val_pred_nb)
print("MAE en validación (Naive Bayes):", round(mae_nb_val, 4))

In [ ]:
# Random Search para Naive Bayes
from scipy.stats import loguniform

param_dist_nb = {'var_smoothing': loguniform(1e-12, 1e-8)}
random_nb = RandomizedSearchCV(GaussianNB(), param_dist_nb, n_iter=15, cv=5, 
                                scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
random_nb.fit(X_train, y_train_binned)

print("Mejores parámetros (RandomSearch):", random_nb.best_params_)
print("Mejor CV score (MAE):", -random_nb.best_score_)

# Criterio de calidad: elegir el que mejor MAE en validación tenga
y_val_pred_nb_rs = nb_pred_to_regression(random_nb.best_estimator_, X_val, bin_centers)
mae_nb_rs_val = mean_absolute_error(y_val, y_val_pred_nb_rs)
print("MAE validación (RandomSearch):", round(mae_nb_rs_val, 4))

if mae_nb_rs_val < mae_nb_val:
    best_nb = random_nb.best_estimator_
    print("Selección: modelo obtenido por RandomSearch")
else:
    best_nb = grid_nb.best_estimator_
    print("Selección: modelo obtenido por GridSearch")

## 6. Modelo 2: KNN

**KNeighborsRegressor** con GridSearch y RandomizedSearch y **Cross Validation**. Criterio de calidad: MAE en validación.

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

param_grid_knn = {
    'n_neighbors': list(range(3, 51, 2)),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}

grid_knn = GridSearchCV(KNeighborsRegressor(), param_grid_knn, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_knn.fit(X_train, y_train)

print("Mejores parámetros (GridSearch):", grid_knn.best_params_)
print("Mejor CV MAE:", -grid_knn.best_score_)

y_val_pred_knn = grid_knn.predict(X_val)
mae_knn_val = mean_absolute_error(y_val, y_val_pred_knn)
print("MAE en validación (KNN):", round(mae_knn_val, 4))

In [ ]:
from scipy.stats import randint

param_dist_knn = {
    'n_neighbors': randint(3, 51),
    'weights': ['uniform', 'distance'],
    'p': [1, 2]
}
random_knn = RandomizedSearchCV(KNeighborsRegressor(), param_dist_knn, n_iter=40, cv=5,
                                 scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
random_knn.fit(X_train, y_train)

print("Mejores parámetros (RandomSearch):", random_knn.best_params_)
y_val_pred_knn_rs = random_knn.predict(X_val)
mae_knn_rs_val = mean_absolute_error(y_val, y_val_pred_knn_rs)
print("MAE validación (RandomSearch):", round(mae_knn_rs_val, 4))

if mae_knn_rs_val < mae_knn_val:
    best_knn = random_knn.best_estimator_
else:
    best_knn = grid_knn.best_estimator_

## 7. Modelo 3: Random Forest (elección libre)

**Random Forest Regressor** con GridSearch y RandomizedSearch y Cross Validation.

In [ ]:
from sklearn.ensemble import RandomForestRegressor

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_rf = GridSearchCV(RandomForestRegressor(random_state=42), param_grid_rf, cv=5, 
                        scoring='neg_mean_absolute_error', n_jobs=-1)
grid_rf.fit(X_train, y_train)

print("Mejores parámetros (GridSearch):", grid_rf.best_params_)
print("Mejor CV MAE:", -grid_rf.best_score_)

y_val_pred_rf = grid_rf.predict(X_val)
mae_rf_val = mean_absolute_error(y_val, y_val_pred_rf)
print("MAE en validación (RF):", round(mae_rf_val, 4))

In [ ]:
param_dist_rf = {
    'n_estimators': randint(50, 250),
    'max_depth': [5, 10, 15, 20, None],
    'min_samples_split': randint(2, 15),
    'min_samples_leaf': randint(1, 8)
}
random_rf = RandomizedSearchCV(RandomForestRegressor(random_state=42), param_dist_rf, n_iter=25, cv=5,
                                scoring='neg_mean_absolute_error', random_state=42, n_jobs=-1)
random_rf.fit(X_train, y_train)

print("Mejores parámetros (RandomSearch):", random_rf.best_params_)
y_val_pred_rf_rs = random_rf.predict(X_val)
mae_rf_rs_val = mean_absolute_error(y_val, y_val_pred_rf_rs)
print("MAE validación (RandomSearch):", round(mae_rf_rs_val, 4))

if mae_rf_rs_val < mae_rf_val:
    best_rf = random_rf.best_estimator_
else:
    best_rf = grid_rf.best_estimator_

## 8. Comparativa de modelos (gráficos)

Integración de gráficos para comparar el entrenamiento y rendimiento de los modelos.

In [ ]:
# MAE en validación por modelo
mae_nb_final = mean_absolute_error(y_val, nb_pred_to_regression(best_nb, X_val, bin_centers))
mae_knn_final = mean_absolute_error(y_val, best_knn.predict(X_val))
mae_rf_final = mean_absolute_error(y_val, best_rf.predict(X_val))

modelos = ['Naive Bayes', 'KNN', 'Random Forest']
maes = [mae_nb_final, mae_knn_final, mae_rf_final]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(modelos, maes, color=['#2ecc71', '#3498db', '#9b59b6'])
ax.set_ylabel('MAE (validación)')
ax.set_title('Comparativa de modelos: MAE en conjunto de validación')
for b, v in zip(bars, maes):
    ax.text(b.get_x() + b.get_width()/2, b.get_height() + 0.2, f'{v:.2f}', ha='center', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# Cross Validation scores (comparativa)
cv_nb = -cross_val_score(best_nb, X_train, y_train_binned, cv=5, scoring='neg_mean_absolute_error')
cv_nb_reg = np.array([mean_absolute_error(y_train.iloc[tt], bin_centers[np.clip(best_nb.predict(X_train.iloc[tt]), 0, len(bin_centers)-1).astype(int)]) for tt in [np.arange(len(y_train))[i] for i in np.array_split(np.arange(len(y_train)), 5)])
from sklearn.model_selection import KFold
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_nb_mae = []
for tr, te in kf.split(X_train):
    y_te_bin = pd.cut(y_train.iloc[te], bins=n_bins, labels=False)
    pred = nb_pred_to_regression(best_nb, X_train.iloc[te], bin_centers)
    cv_nb_mae.append(mean_absolute_error(y_train.iloc[te], pred))
cv_knn = -cross_val_score(best_knn, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')
cv_rf = -cross_val_score(best_rf, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')

fig, ax = plt.subplots(figsize=(9, 5))
ax.boxplot([cv_nb_mae, cv_knn, cv_rf], labels=modelos, patch_artist=True)
ax.set_ylabel('MAE')
ax.set_title('Distribución de MAE en Cross Validation (5 folds)')
plt.tight_layout()
plt.show()

## 9. Predicción y validación de resultados

Uso de **herramientas gráficas** para entender la precisión y **descripción clara** de la validación.

In [ ]:
# Evaluación en conjunto de test (simulando evaluación final)
y_test_pred_nb = nb_pred_to_regression(best_nb, X_test, bin_centers)
y_test_pred_knn = best_knn.predict(X_test)
y_test_pred_rf = best_rf.predict(X_test)

mae_test_nb = mean_absolute_error(y_test, y_test_pred_nb)
mae_test_knn = mean_absolute_error(y_test, y_test_pred_knn)
mae_test_rf = mean_absolute_error(y_test, y_test_pred_rf)

print("MAE en conjunto TEST:")
print("  Naive Bayes:", round(mae_test_nb, 4))
print("  KNN:", round(mae_test_knn, 4))
print("  Random Forest:", round(mae_test_rf, 4))

# Modelo elegido para submit: el de menor MAE en test
best_model_name = min([('Naive Bayes', mae_test_nb), ('KNN', mae_test_knn), ('Random Forest', mae_test_rf)], key=lambda x: x[1])
print("\nModelo seleccionado para la competición:", best_model_name[0])

In [ ]:
# Gráfico: Valores reales vs predichos (test)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name, y_pred in zip(axes, modelos, [y_test_pred_nb, y_test_pred_knn, y_test_pred_rf]):
    ax.scatter(y_test, y_pred, alpha=0.5, s=20)
    ax.plot([0, y_test.max()], [0, y_test.max()], 'r--', label='Ideal')
    ax.set_xlabel('Real')
    ax.set_ylabel('Predicho')
    ax.set_title(name)
    ax.legend()
plt.suptitle('Precisión de los resultados: Real vs Predicho (conjunto test)')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución de errores (residuos)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, name, y_pred in zip(axes, modelos, [y_test_pred_nb, y_test_pred_knn, y_test_pred_rf]):
    resid = y_test.values - y_pred
    ax.hist(resid, bins=30, edgecolor='black', alpha=0.7)
    ax.axvline(0, color='red', linestyle='--')
    ax.set_xlabel('Error (real - predicho)')
    ax.set_title(name)
plt.suptitle('Distribución de errores en test')
plt.tight_layout()
plt.show()

### Descripción de la validación

- **Conjunto de validación**: usado para elegir hiperparámetros (GridSearch/RandomSearch) y para comparar modelos sin tocar el test.
- **Conjunto de test**: usado una sola vez para reportar MAE final y elegir el modelo a subir.
- **Métrica**: MAE (Mean Absolute Error), coherente con la valoración de la competición DrivenData.
- Los gráficos Real vs Predicho muestran si hay sesgo (sobre/subestimación) y la dispersión del error.

## 10. Propuestas creativas e innovadoras

- **Uso de validación explícita:** División train/validation/test permite elegir hiperparámetros y el modelo final sin sobreajustar al test.
- **Selección de características con gráficos:** Correlación con el target, importancia con árbol de decisión y matriz de correlación para evitar redundancia.
- **Naive Bayes en regresión:** Discretización del target en intervalos para usar GaussianNB y conversión de la clase predicha al valor central del intervalo.
- **Dos estrategias de búsqueda:** GridSearch y RandomizedSearch con Cross Validation para justificar la elección del modelo por MAE.
- **Posible extensión:** Entrenar modelos separados por ciudad (sj/iq) o añadir características temporales (lag de casos, media móvil) para capturar estacionalidad.

## 11. Submit: fichero de predicción para la competición

Se genera el CSV con el formato requerido por DrivenData. Incluir **captura de la valoración/posicionamiento** en la web tras subir el fichero.

In [ ]:
# Predicción sobre el test de la competición
X_test_comp = X_test_norm[selected_features]

if best_model_name[0] == 'Naive Bayes':
    pred_submit = nb_pred_to_regression(best_nb, X_test_comp, bin_centers)
elif best_model_name[0] == 'KNN':
    pred_submit = best_knn.predict(X_test_comp)
else:
    pred_submit = best_rf.predict(X_test_comp)

# Valores enteros no negativos (total_cases)
pred_submit = np.round(np.maximum(0, pred_submit)).astype(int)

# Formato DrivenData: city, year, weekofyear, total_cases
submit = features_test[['city', 'year', 'weekofyear']].copy()
submit['total_cases'] = pred_submit

out_path = 'submission.csv'
submit.to_csv(out_path, index=False)
print("Fichero guardado:", out_path)
print(submit.head(10))

### Instrucciones para el submit
1. Entra en https://www.drivendata.org/competitions/44/dengai-predicting-disease-spread/
2. Inicia sesión y ve a "Submit"
3. Sube el fichero `submission.csv`
4. Realiza una **captura de pantalla** de la valoración (MAE) y posicionamiento en el leaderboard para incluirla en el PDF.

---
## Conclusiones

- Se ha realizado la importación del dataset desde carpeta local (equivalente a GitHub/Drive), preparación (normalización, tratamiento de faltantes), selección de características con correlación e importancia, y división train/validation/test.
- Se han entrenado tres modelos (Naive Bayes vía discretización, KNN y Random Forest) con GridSearch y RandomizedSearch y Cross Validation, justificando la selección por MAE en validación.
- Los gráficos permiten comparar modelos y entender la precisión (real vs predicho, distribución de errores).
- El fichero `submission.csv` está listo para subir a DrivenData; es obligatorio incluir en el PDF la captura del resultado y posicionamiento en la competición.

## Referencias

- DrivenData - DengAI: Predicting Disease Spread. https://www.drivendata.org/competitions/44/dengai-predicting-disease-spread/
- scikit-learn: Machine Learning in Python. https://scikit-learn.org/
- Pandas documentation. https://pandas.pydata.org/docs/
- Planetachatbot - Modelos de machine learning: Guía básica. https://planetachatbot.com/modelos-de-machine-learning-guia-basica-para-principiantes/
- Brownlee, J. - How to Use Correlation to Understand the Relationship Between Variables. Machine Learning Mastery. https://machinelearningmastery.com/ (referencia externa no recogida en el material suministrado).